# `RunnableLambda: Runnable[Input, Output]`

`RunnableLambda` converts a Python callable into a `Runnable`.

It supports synchronous and asynchronous execution, LangChain tracing, batching, composition, and optional generator-based output.

For components designed primarily for true streaming transformations, use `RunnableGenerator`.

## Type Parameters

```python
Input # Input type accepted by the callable
Output # Output type returned or yielded by the callable
```

## Fields

```python
func: Callable[..., Output] # Synchronous callable used for synchronous execution
afunc: Callable[..., Awaitable[Output] | AsyncIterator[Output]] # Asynchronous callable used for asynchronous execution
name: str | None # Runnable name used for tracing and debugging
```

Only the callable forms supplied to the constructor are stored.

## Constructor

```python
RunnableLambda(
    func: Callable[..., Output | Iterator[Output] | Awaitable[Output] | AsyncIterator[Output] | Runnable], # Synchronous or asynchronous callable converted into a Runnable
    afunc: Callable[..., Awaitable[Output] | AsyncIterator[Output]] | None = None, # Optional native asynchronous implementation
    name: str | None = None, # Optional custom Runnable name
) -> None # Initialize the RunnableLambda
```

The callable may optionally receive `RunnableConfig`, a callback manager, or both in addition to the input.

The constructor raises `TypeError` when `func` is not callable or when both `func` and `afunc` are asynchronous callables.

## Callable Forms

```python
Callable[[Input], Output] # Regular synchronous callable
Callable[[Input, RunnableConfig], Output] # Synchronous callable receiving runtime configuration
Callable[[Input, CallbackManagerForChainRun], Output] # Synchronous callable receiving a callback manager
Callable[[Input, CallbackManagerForChainRun, RunnableConfig], Output] # Synchronous callable receiving callbacks and configuration
Callable[[Input], Iterator[Output]] # Synchronous generator callable
Callable[[Input], Awaitable[Output]] # Asynchronous callable
Callable[[Input], AsyncIterator[Output]] # Asynchronous generator callable
Callable[[Input], Runnable] # Callable returning another Runnable
```

## Developer-Facing Properties and Methods

### `InputType`

Infers the input type from the first parameter annotation of the wrapped callable.

Returns `Any` when the type cannot be inferred.

### `get_input_schema`

Returns a Pydantic schema based on the inferred input type.

It can also infer dictionary fields used by `itemgetter` or keys accessed from the first dictionary argument.

### `OutputType`

Infers the output type from the callable's return annotation.

For iterator and asynchronous-iterator annotations, it returns the yielded item type.

Returns `Any` when the type cannot be inferred.

### `get_output_schema`

Returns a Pydantic schema based on the inferred output type and the module containing the wrapped callable.

### `deps`

Returns `Runnable` objects captured as nonlocal dependencies by the wrapped callable.

### `config_specs`

Returns the combined configurable-field specifications of the discovered Runnable dependencies.

### `get_graph`

Returns a graph containing discovered Runnable dependencies.

When no dependencies are found, it returns the standard Runnable graph.

### `__eq__`

Returns `True` when two `RunnableLambda` objects wrap the same synchronous callable or the same asynchronous callable.

### `__repr__`

Returns a readable representation containing the wrapped function or lambda source when available.

### `invoke`

Synchronously executes the wrapped synchronous callable.

It raises `TypeError` when the instance contains only an asynchronous callable.

When a generator function is wrapped, all generated chunks are combined into the final result.

When the callable returns another `Runnable`, that Runnable is invoked using the original input.

### `ainvoke`

Asynchronously executes the supplied asynchronous callable.

When no asynchronous callable is supplied, it executes the synchronous callable through an executor.

When an asynchronous generator is wrapped, all generated chunks are combined into the final result.

When the callable returns another `Runnable`, that Runnable is invoked asynchronously using the original input.

### `transform`

Consumes a synchronous input iterator before executing the wrapped synchronous callable.

If the input chunks support addition, they are combined; otherwise, the final input chunk is used.

A generator callable yields its chunks individually.

It raises `TypeError` when the instance contains only an asynchronous callable.

### `stream`

Wraps one input value as an iterator and delegates to `transform()`.

A regular callable normally produces one output chunk.

A generator callable may produce multiple output chunks.

### `atransform`

Consumes an asynchronous input iterator before executing the asynchronous implementation.

If no asynchronous callable is supplied, the synchronous callable is executed through an executor.

An asynchronous generator yields its chunks individually.

### `astream`

Wraps one input value as an asynchronous iterator and delegates to `atransform()`.

A regular callable normally produces one output chunk.

An asynchronous generator may produce multiple output chunks.

## Returned Runnable Behaviour

When the wrapped callable returns another `Runnable`, `RunnableLambda` automatically executes that returned Runnable with the original input.

The configured recursion limit prevents unlimited nested Runnable execution.

## Important Behaviour

- A synchronous callable supports `invoke()` and `ainvoke()`.
- An asynchronous-only callable supports `ainvoke()` but not `invoke()`.
- Providing `afunc` allows separate optimized sync and async implementations.
- Type annotations improve generated input and output schemas.
- Generator chunks are combined for `invoke()` and `ainvoke()`.
- Generator chunks are emitted individually by `stream()` and `astream()`.
- The class is unhashable because equality is based on the wrapped callable.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import RunnableLambda

def calculate_square(number: int) -> int: # Define a function that accepts a number
    return number ** 2 # Return the square of the number

square_runnable = RunnableLambda(calculate_square) # Convert the function into a Runnable

result = square_runnable.invoke(5) # Execute the Runnable with input 5

print(result) # Display the returned result